# Manual Bible ASR Fine-tuning (No API Key)

This notebook demonstrates how to fine-tune the Runyoro HuBERT-style model using manually downloaded Bible audio and text. It mirrors the workflow from `1_data_download_and_manifest_creation.ipynb` and `runyoro_bible_asr_finetune.ipynb` but avoids the Digital Bible Platform API.


## Install Dependencies
Run this once to install the libraries required for scraping, audio handling and SpeechBrain.


In [ ]:
!pip install requests beautifulsoup4 soundfile torch torchaudio speechbrain PyYAML

## Path Setup

In [ ]:
import subprocess, osfrom pathlib import Pathtry:    repo_root = subprocess.run(        ['git', 'rev-parse', '--show-toplevel'],        capture_output=True, text=True, check=True    ).stdout.strip()    PROJECT_ROOT = Path(repo_root)except Exception:    PROJECT_ROOT = Path.cwd()  # fallback# You can manually adjust PROJECT_ROOT here if the repo is elsewhere.DATA_DIR = PROJECT_ROOT / 'data' / 'bible_manual'AUDIO_DIR = DATA_DIR / 'audio'TEXT_DIR = DATA_DIR / 'text'AUDIO_FILESET_ID = 'NYOBSUN2DA'TEXT_FILESET_ID = 'NYOBSUN2DA_TEXT'AUDIO_DIR.mkdir(parents=True, exist_ok=True)TEXT_DIR.mkdir(parents=True, exist_ok=True)os.chdir(PROJECT_ROOT)print('Project root:', PROJECT_ROOT)print('Data dir:', DATA_DIR)

## Download Audio Zip
The audio is hosted as a pre-zipped archive. Update `audio_zip_url` if the link expires.


In [ ]:
import requests, zipfile
zip_path = AUDIO_DIR / f'{AUDIO_FILESET_ID}.zip'
audio_zip_url = 'https://d1gd73roq7kqw6.cloudfront.net/audio/NYOBSU/NYOBSUN2DA/NYOBSUN2DA.zip?x-amz-transaction=6944150&Expires=1749382538&Signature=X1IFdAguEgRkF4SIG5bmdFjfTaOCb6UHQ6NDonTvJznDmaYPkEBVU2v9zT87IN5GDYzPqv3Eom~L1vPEh-YrIzDNAIecnePJAUJ96VEaVrOFE9mudBEjSMLFI5m6Gzm4YaHgx1g7SEPSP5QIu6bTqkzB2d5fCTJsB9Yt9O6u2mKx9j3zloo9gc43qyKW-SzJMfWbZhBkEPR~dkKA3my5SDVVeO21ZpODgIp1O9TETjVTx6--fauwoxpbMgI9NXTUvFWt2zD7BWVxgkPAY2IGDuKoX~H4xNJwn7eF4TZeqK3Udkx7LrM5WmNVMfOHTN5E68vWERY37l4kYZpP~pIB6g__&Key-Pair-Id=APKAI4ULLVMANLYYPTLQ'

if not zip_path.exists():
    print('Downloading audio zip...')
    with requests.get(audio_zip_url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
else:
    print('Audio zip already present.')

extract_dir = AUDIO_DIR / AUDIO_FILESET_ID
if not extract_dir.exists():
    print('Extracting zip...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
else:
    print('Audio already extracted.')
print('Audio ready in', extract_dir)


## Scrape Text from Bible.is
Parse the audio filenames to know which book and chapter pairs exist, then fetch matching text for each one.


In [ ]:
import requests
import re
from collections import defaultdict
from bs4 import BeautifulSoup

text_base = TEXT_DIR / TEXT_FILESET_ID
text_base.mkdir(parents=True, exist_ok=True)

headers = {'User-Agent': 'Mozilla/5.0'}

def fetch_chapter(book, chapter):
    url = f'https://live.bible.is/bible/NYOBSU/{book}/{chapter}'
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    text = soup.get_text(separator=' ')
    text = ' '.join(text.split())
    return text

# determine available books/chapters from audio files
audio_dir = AUDIO_DIR / AUDIO_FILESET_ID
books = defaultdict(set)
for f in audio_dir.glob("*.*"):
    m = re.match(rf"{AUDIO_FILESET_ID}_(\w+)_(\d+)", f.stem)
    if m:
        book_id, chapter = m.groups()
        books[book_id].add(int(chapter))

for book_id, chapters in books.items():
    for ch in sorted(chapters):
        txt = fetch_chapter(book_id, ch)
        out_path = text_base / f"{book_id}_{ch:03d}.txt"
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(txt)
        print('Wrote', out_path)


## Build SpeechBrain Manifest

In [ ]:
manifest_path = DATA_DIR / 'manual_manifest.json'
!python runyoro_speech_ai/asr_finetune/build_manifest.py     --audio_dir_base {AUDIO_DIR}     --text_dir_base {TEXT_DIR}     --audio_fileset_id {AUDIO_FILESET_ID}     --text_fileset_id {TEXT_FILESET_ID}     --manifest_path {manifest_path}     --language_code nyo


## Prepare Hyperparameters

In [ ]:
import yaml
hparams_dir = PROJECT_ROOT / 'asr_manual_finetune'
hparams_dir.mkdir(parents=True, exist_ok=True)
hparams_file = hparams_dir / 'hparams_finetune.yaml'

hparams = {
    'seed': 1234,
    'data_folder': str(DATA_DIR),
    'output_folder': str(hparams_dir),
    'save_folder': '!ref <output_folder>/save',
    'train_manifest': manifest_path.name,
    'valid_manifest': manifest_path.name,
    'test_manifest': manifest_path.name,
    'tokenizer_model_dir': '!ref <save_folder>/tokenizer/',
    'tokenizer_model_prefix': 'spm_unigram_1000',
    'tokenizer_vocab_size': 1000,
    'ssl_model_hub': 'facebook/wav2vec2-base',
    'ssl_local_checkpoint_path': '/path/to/ssl_checkpoint.ckpt',
    'number_of_epochs': 5,
}
with open(hparams_file, 'w') as f:
    yaml.dump(hparams, f, sort_keys=False)
print('Hparams saved to', hparams_file)


## Start ASR Fine-tuning

In [ ]:
!python runyoro_speech_ai/asr_finetune/train_ctc.py {hparams_file}
